# Analyse exploratoire des accidents de la route – BAAC 2024

## Objectif
Explorer et préparer les données BAAC 2024 afin de comprendre leur structure,
leur qualité et leur potentiel analytique avant toute interprétation.

In [ ]:
from pathlib import Path
from src.io import load_baac_csv

RAW = Path("../data/raw")

caract = load_baac_csv(RAW / "caract-2024.csv")
print("caract chargé :", caract.shape)


caract chargé : (54402, 15)


In [3]:
caract.head()
caract.info()
caract.duplicated().sum()
caract.isna().mean().sort_values(ascending=False).head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54402 entries, 0 to 54401
Data columns (total 15 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Num_Acc  54402 non-null  int64 
 1   jour     54402 non-null  int64 
 2   mois     54402 non-null  int64 
 3   an       54402 non-null  int64 
 4   hrmn     54402 non-null  object
 5   lum      54402 non-null  int64 
 6   dep      54402 non-null  object
 7   com      54402 non-null  object
 8   agg      54402 non-null  int64 
 9   int      54402 non-null  int64 
 10  atm      54402 non-null  int64 
 11  col      54402 non-null  int64 
 12  adr      52092 non-null  object
 13  lat      54402 non-null  object
 14  long     54402 non-null  object
dtypes: int64(9), object(6)
memory usage: 6.2+ MB


adr        0.042462
jour       0.000000
Num_Acc    0.000000
an         0.000000
hrmn       0.000000
lum        0.000000
mois       0.000000
dep        0.000000
com        0.000000
int        0.000000
dtype: float64

In [ ]:
lieux = load_baac_csv(RAW / "lieux-2024.csv")
print("lieux chargé :", lieux.shape)

lieux chargé : (70248, 18)


In [5]:
lieux.head()
lieux.info()
lieux.duplicated().sum()
lieux.isna().mean().sort_values(ascending=False).head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70248 entries, 0 to 70247
Data columns (total 18 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Num_Acc  70248 non-null  int64 
 1   catr     70248 non-null  int64 
 2   voie     56917 non-null  object
 3   v1       70248 non-null  int64 
 4   v2       5916 non-null   object
 5   circ     70248 non-null  int64 
 6   nbv      70248 non-null  object
 7   vosp     70248 non-null  int64 
 8   prof     70248 non-null  int64 
 9   pr       70248 non-null  object
 10  pr1      70248 non-null  object
 11  plan     70248 non-null  int64 
 12  lartpc   33 non-null     object
 13  larrout  70248 non-null  object
 14  surf     70248 non-null  int64 
 15  infra    70248 non-null  int64 
 16  situ     70248 non-null  int64 
 17  vma      70248 non-null  int64 
dtypes: int64(11), object(7)
memory usage: 9.6+ MB


lartpc     0.999530
v2         0.915784
voie       0.189771
Num_Acc    0.000000
catr       0.000000
v1         0.000000
nbv        0.000000
vosp       0.000000
prof       0.000000
circ       0.000000
dtype: float64

In [6]:
print("Num_Acc dans caract :", "Num_Acc" in caract.columns)
print("Num_Acc dans lieux  :", "Num_Acc" in lieux.columns)
caract["Num_Acc"].nunique(), lieux["Num_Acc"].nunique()

Num_Acc dans caract : True
Num_Acc dans lieux  : True


(54402, 54402)

In [7]:
acc = caract.merge(
    lieux,
    on="Num_Acc",
    how="left",
    suffixes=("", "_lieux")
)

print("Table accidents fusionnée :", acc.shape)
acc.head()

Table accidents fusionnée : (70248, 32)


,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,...,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
0,202400000001,25,3,2024,07:40,2,70,70285,1,1,...,1,1,260,2,NaN,7,1,0,1,90
1,202400000002,20,3,2024,15:05,1,21,21054,2,3,...,1,-1,-1,1,NaN,-1,9,0,1,30
2,202400000002,20,3,2024,15:05,1,21,21054,2,3,...,1,-1,-1,1,NaN,-1,9,0,1,30
3,202400000003,22,3,2024,19:30,2,15,15012,1,1,...,1,-1,-1,1,NaN,-1,1,0,3,50
4,202400000004,24,3,2024,17:50,1,14,14118,2,3,...,1,-1,-1,1,NaN,-1,1,9,1,50


In [ ]:
usag = load_baac_csv(
    RAW / "usagers-2024.csv",
    usecols=["Num_Acc", "grav"]
)
print("usagers chargé :", usag.shape)

usagers chargé : (125187, 2)


In [9]:
usag.head()
usag.info()
usag.duplicated().sum()
usag.isna().mean().sort_values(ascending=False).head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125187 entries, 0 to 125186
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   Num_Acc  125187 non-null  int64
 1   grav     125187 non-null  int64
dtypes: int64(2)
memory usage: 1.9 MB


Num_Acc    0.0
grav       0.0
dtype: float64

In [10]:
grav_map = {
    1: "Indemne",
    2: "Tué",
    3: "Blessé hospitalisé",
    4: "Blessé léger"
}

usag["grav_label"] = usag["grav"].map(grav_map)
usag["grav_label"].value_counts()


grav_label
Indemne               52920
Blessé léger          49709
Blessé hospitalisé    19126
Tué                    3432
Name: count, dtype: int64

In [11]:
# conversion "HH:MM" -> datetime puis extraction de l'heure
acc["hour"] = pd.to_datetime(acc["hrmn"], format="%H:%M", errors="coerce").dt.hour

# contrôle rapide
acc[["hrmn", "hour"]].head(10)
acc.columns

Index(['Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg',
       'int', 'atm', 'col', 'adr', 'lat', 'long', 'catr', 'voie', 'v1', 'v2',
       'circ', 'nbv', 'vosp', 'prof', 'pr', 'pr1', 'plan', 'lartpc', 'larrout',
       'surf', 'infra', 'situ', 'vma', 'hour'],
      dtype='object')

In [12]:
# création de la date
acc["date"] = pd.to_datetime(
    acc["an"].astype(str) + "-" +
    acc["mois"].astype(str).str.zfill(2) + "-" +
    acc["jour"].astype(str).str.zfill(2),
    errors="coerce"
)

# création du jour de la semaine
acc["weekday"] = acc["date"].dt.day_name()
acc[["date", "weekday"]].head()

# création de l'heure (hrmn est au format "HH:MM")
acc["hour"] = pd.to_datetime(acc["hrmn"], format="%H:%M", errors="coerce").dt.hour
acc[["hrmn", "hour"]].head()

,hrmn,hour
0,07:40,7
1,15:05,15
2,15:05,15
3,19:30,19
4,17:50,17


In [13]:
acc["hour"].value_counts().sort_index().head()
acc["weekday"].value_counts()

weekday
Friday       11638
Thursday     10583
Wednesday    10105
Tuesday      10051
Saturday      9858
Monday        9669
Sunday        8344
Name: count, dtype: int64

In [14]:
from pathlib import Path

PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)
print("OK processed folder:", PROCESSED.resolve())


OK processed folder: C:\Users\Hannae\Documents\Projets\accidents-france-baac\data\processed


In [15]:
acc.to_parquet(PROCESSED / "accidents-2024.parquet", index=False)
usag.to_parquet(PROCESSED / "usagers-2024.parquet", index=False)

print("Parquet générés ✅")


Parquet générés ✅


In [16]:
list(PROCESSED.glob("*.parquet"))


[WindowsPath('../data/processed/accidents-2024.parquet'),
 WindowsPath('../data/processed/usagers-2024.parquet')]

## Conclusion de l’analyse exploratoire

Les données BAAC 2024 présentent une structure exploitable pour l’analyse des accidents. Les principales variables temporelles et territoriales sont disponibles, bien que certaines valeurs manquantes et codifications nécessitent une interprétation prudente. Les données préparées permettent désormais de conduire une analyse orientée insights.